In [41]:
import xarray as xr

HAZARD_ZELL = "/home/gespejogutierrez/Lisflood_climada/Zell_netcdfiles/Zell_2m_COSMO_2022-05-05T12-00.nc"

# Open with xarray (best for analysis)
Zell_Forecast = xr.open_dataset(HAZARD_ZELL)
Zell_Forecast

<xarray.Dataset>
Dimensions:                  (forecast_reference_time: 1, lead_time: 11,
                              realization: 11, x: 2500, y: 1500)
Coordinates:
  * forecast_reference_time  (forecast_reference_time) datetime64[ns] 2022-05...
  * lead_time                (lead_time) float64 0.0 3.6e+03 ... 3.6e+04
  * realization              (realization) int32 0 1 2 3 4 5 6 7 8 9 10
  * x                        (x) float64 2.702e+06 2.702e+06 ... 2.707e+06
  * y                        (y) float64 1.258e+06 1.258e+06 ... 1.255e+06
Data variables:
    time                     (forecast_reference_time, lead_time) datetime64[ns] ...
    water_depth              (forecast_reference_time, lead_time, realization, y, x) float32 ...
    vel_x_c                  (forecast_reference_time, lead_time, realization, y, x) float32 ...
    vel_y_c                  (forecast_reference_time, lead_time, realization, y, x) float32 ...
    flux_x_c                 (forecast_reference_time, lead_time, realization, y, x) float32 ...
    flux_y_c                 (forecast_reference_time, lead_time, realization, y, x) float32 ...
    vel_mag                  (forecast_reference_time, lead_time, realization, y, x) float32 ...
    flux_mag                 (forecast_reference_time, lead_time, realization, y, x) float32 ...
Attributes:
    Conventions:  CF-1.8
    source:       LISFLOOD ASCII ensemble stacked to COSMO-like forecast form...
    crs:          EPSG:2056

In [43]:
# ============================================================
# Build CLIMADA Hazard objects from LISFLOOD-FP ensemble forecast
# - Works with forecast_reference_time dimension (new format)
# - Keeps ens_variables structure
# - Builds sparse intensity using ONLY wet cells (memory safe)
# - Creates one Hazard per lead_time (each with n_ens events)
# ============================================================

from climada.hazard import Hazard
from climada.hazard.centroids.centr import Centroids
from pyproj import Transformer
from scipy import sparse
import numpy as np
import xarray as xr
import pandas as pd
from datetime import timedelta
import gc

# ----------------------------
# INPUTS
# ----------------------------
HAZARD_ZELL = "/home/gespejogutierrez/Lisflood_climada/Zell_netcdfiles/Zell_2m_COSMO_2022-05-05T12-00.nc"
VAR_NAME    = "water_depth"
GRID_EPSG   = 2056

# choose your wet threshold (m) - you mentioned 0.04
WET_THRESH  = 0.01

FRT_DIM     = "forecast_reference_time"
ENS_DIM     = "realization"
TIME_DIM    = "lead_time"
EVENT_NAME  = "ZELL"

# ----------------------------
# READ DATASET
# ----------------------------
ds = xr.open_dataset(HAZARD_ZELL)

# Get init/forecast reference time
if (FRT_DIM in ds.dims) or (FRT_DIM in ds.coords):
    init_time = pd.to_datetime(ds[FRT_DIM].values[0])
else:
    # fallback if missing
    init_time = pd.to_datetime("2022-05-05T12:00")

# Select first forecast_reference_time slice if present
ds0 = ds.isel({FRT_DIM: 0}) if (FRT_DIM in ds.dims) else ds

# Pull variable and enforce dim order: (realization, lead_time, y, x)
da0 = ds0[VAR_NAME].transpose(ENS_DIM, TIME_DIM, "y", "x")

# Sizes
n_ens  = ds0.sizes[ENS_DIM]
n_time = ds0.sizes[TIME_DIM]
ny     = ds0.sizes["y"]
nx     = ds0.sizes["x"]
n_cent = ny * nx

print(f"Dataset: {n_ens} ens × {n_time} lead × {ny}×{nx} grid")
print(f"Centroids: {n_cent}")

# ----------------------------
# CENTROIDS (once)
# ----------------------------
x = ds0["x"].values
y = ds0["y"].values
X, Y = np.meshgrid(x, y)

transformer = Transformer.from_crs(GRID_EPSG, 4326, always_xy=True)
lon, lat = transformer.transform(X.ravel(order="C"), Y.ravel(order="C"))

cent = Centroids.from_lat_lon(lat=lat, lon=lon)
print(f"Created {cent.size} centroids")

# ----------------------------
# LEAD TIMES: seconds -> hours if needed
# ----------------------------
lead_vals = ds0[TIME_DIM].values.astype(float)
lead_hours = lead_vals / 3600.0 if np.nanmax(lead_vals) > 1000 else lead_vals
print("Lead times (hours):", lead_hours)

# ----------------------------
# STEP 1: BUILD ens_variables (kept, as you requested)
# ----------------------------
ens_variables = {}

class LeadTimeData:
    """Container to mimic ens_variables['t_idx'].data[ens_idx].attrs"""
    def __init__(self, data_list, run_datetime):
        self.data = data_list
        self.attrs = {"run_datetime": run_datetime}

for t_idx in range(n_time):
    run_dt = init_time + timedelta(hours=float(lead_hours[t_idx]))
    data_list = []
    for e_idx in range(n_ens):
        da_sel = da0.isel({ENS_DIM: e_idx, TIME_DIM: t_idx})  # (y, x)
        # attach attrs (lightweight)
        da_sel.attrs["run_datetime"] = run_dt
        da_sel.attrs["ensemble_member"] = e_idx
        da_sel.attrs["lead_time_hr"] = float(lead_hours[t_idx])
        data_list.append(da_sel)
    ens_variables[str(t_idx)] = LeadTimeData(data_list, run_dt)

print(f"\nCreated ens_variables with {len(ens_variables)} lead times and {n_ens} ensembles each.")

# ----------------------------
# STEP 2: BUILD Hazard objects per lead time (sparse, wet-only)
# ----------------------------
haz_names = {}

for t_idx in range(n_time):
    da_lead = ens_variables[str(t_idx)]
    run_dt = da_lead.attrs["run_datetime"]

    rows_all = []
    cols_all = []
    data_all = []

    # Build sparse COO pieces: each ensemble is one event (row index)
    for e_idx in range(n_ens):
        arr = da_lead.data[e_idx].values  # (y, x) float32
        flat = arr.ravel(order="C")

        wet = flat > WET_THRESH
        idx = np.nonzero(wet)[0]

        if idx.size == 0:
            continue

        rows_all.append(np.full(idx.shape, e_idx, dtype=np.int32))
        cols_all.append(idx.astype(np.int32, copy=False))
        data_all.append(flat[idx].astype(np.float32, copy=False))

    if rows_all:
        rows = np.concatenate(rows_all)
        cols = np.concatenate(cols_all)
        data = np.concatenate(data_all)
        intensity_sparse = sparse.csr_matrix(
            (data, (rows, cols)),
            shape=(n_ens, n_cent),
            dtype=np.float32
        )
    else:
        intensity_sparse = sparse.csr_matrix((n_ens, n_cent), dtype=np.float32)

    # Create Hazard object
    haz_obj = Hazard("FL")
    haz_obj.centroids = cent
    haz_obj.units = "m"
    haz_obj.intensity = intensity_sparse
    haz_obj.event_id = np.arange(1, n_ens + 1, dtype=int)

    haz_obj.event_name = np.array([
        f"{EVENT_NAME}_ens{e+1:02d}_{run_dt.strftime('%Y-%m-%dT%H:%M')}"
        for e in range(n_ens)
    ])
    haz_obj.date = np.array([int(run_dt.strftime("%Y%m%d"))] * n_ens, dtype=int)
    haz_obj.frequency = np.ones(n_ens, dtype=float)

    haz_obj.tag = {
        "lead_time_hr": float(lead_hours[t_idx]),
        "run_datetime": run_dt.strftime("%Y-%m-%dT%H:%M"),
        "forecast_reference_time": str(init_time),
        "wet_threshold": WET_THRESH,
        "source": HAZARD_ZELL
    }

    key_name = f"{EVENT_NAME}_{run_dt.strftime('%Y-%m-%dT%H:%M')}"
    haz_names[key_name] = haz_obj

    # small progress + memory info
    dens = haz_obj.intensity.nnz / (n_ens * n_cent)
    print(f"Lead {t_idx:02d} | {key_name} | nnz={haz_obj.intensity.nnz} | density={dens:.2e}")

    # Optional: free temporary arrays aggressively (helps on UBELIX)
    del rows_all, cols_all, data_all, intensity_sparse
    gc.collect()

print(f"\nCreated {len(haz_names)} Hazard objects (one per lead time).")

# ----------------------------
# STEP 3: Example access
# ----------------------------
sample_key = list(haz_names.keys())[0]
sample_haz = haz_names[sample_key]

print(f"\nExample key: {sample_key}")
print(f"  → Events: {sample_haz.size} (should be {n_ens})")
print(f"  → Intensity shape: {sample_haz.intensity.shape}")
print(f"  → Non-zero cells (nnz): {sample_haz.intensity.nnz}")
print(f"  → Example event names: {sample_haz.event_name[:3]}")

# Optional: check max depth among wet cells for that lead time
if sample_haz.intensity.nnz > 0:
    print(f"  → Max wet depth (m): {sample_haz.intensity.data.max():.3f}")
else:
    print("  → No wet cells above threshold in sample lead time.")


Dataset: 11 ens × 11 lead × 1500×2500 grid
Centroids: 3750000


/tmp/ipykernel_2907286/532601962.py:72: DeprecatedWarning: from_lat_lon is deprecated. This method will be removed in a future version. Simply use the constructor instead.
  cent = Centroids.from_lat_lon(lat=lat, lon=lon)


Created 3750000 centroids
Lead times (hours): [ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10.]

Created ens_variables with 11 lead times and 11 ensembles each.
Lead 00 | ZELL_2022-05-05T12:00 | nnz=0 | density=0.00e+00
Lead 01 | ZELL_2022-05-05T13:00 | nnz=0 | density=0.00e+00
Lead 02 | ZELL_2022-05-05T14:00 | nnz=4168 | density=1.01e-04
Lead 03 | ZELL_2022-05-05T15:00 | nnz=7749 | density=1.88e-04
Lead 04 | ZELL_2022-05-05T16:00 | nnz=7953 | density=1.93e-04
Lead 05 | ZELL_2022-05-05T17:00 | nnz=51914 | density=1.26e-03
Lead 06 | ZELL_2022-05-05T18:00 | nnz=608899 | density=1.48e-02
Lead 07 | ZELL_2022-05-05T19:00 | nnz=1589453 | density=3.85e-02
Lead 08 | ZELL_2022-05-05T20:00 | nnz=2206343 | density=5.35e-02
Lead 09 | ZELL_2022-05-05T21:00 | nnz=2713572 | density=6.58e-02
Lead 10 | ZELL_2022-05-05T22:00 | nnz=3139792 | density=7.61e-02

Created 11 Hazard objects (one per lead time).

Example key: ZELL_2022-05-05T12:00
  → Events: 11 (should be 11)
  → Intensity shape: (11, 3750000)
  →

In [3]:
haz_names ['ZELL_2022-05-05T18:00'].intensity.data 

array([0.043, 0.014, 0.034, ..., 0.142, 0.036, 0.053], dtype=float32)

In [44]:
## EXPOSURE 
import geopandas as gpd
import pandas as pd
from pathlib import Path
import shapely
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from datetime import datetime

# on climada_petals branch feature/networks until merged!
# on climada_python develop branch
from climada.util import coordinates as u_coords
from climada.entity.exposures.base import Exposures
from climada.entity.impact_funcs import ImpactFunc, ImpactFuncSet
from climada.engine import Impact
from climada.hazard.base import Hazard
from climada.util import lines_polys_handler as u_lp
from climada.util.api_client import Client

#from climada_petals.entity.exposures.openstreetmap import osm_dataloader as osm
from climada_petals.entity.exposures import osm_dataloader as osm
from climada.util import coordinates as u_coords
from climada.hazard.base import Hazard

In [45]:
import geopandas as gpd
import xarray as xr
from shapely.geometry import box

BULD_SHP   = "/home/gespejogutierrez/Lisflood_climada/Data_for_process/geo_hwr_footp_tlm2023.shp"
HAZARD_ZELL = "/home/gespejogutierrez/Lisflood_climada/Zell_netcdfiles/Zell_2m_COSMO_2022-05-05T12-00.nc"

# --- 1) Read buildings ---
gdf_build = gpd.read_file(BULD_SHP)
if gdf_build.empty:
    raise ValueError("No polygons found in the shapefile.")

# Ensure CRS is set (LV95 / EPSG:2056 typical)
if gdf_build.crs is None:
    gdf_build = gdf_build.set_crs(2056, allow_override=True)
else:
    gdf_build = gdf_build.to_crs(2056)

# Rename active geometry column
gdf_build = gdf_build.rename_geometry("geom_polygon_lv95")

# --- 2) Get model domain bbox from NetCDF (LV95) ---
ds = xr.open_dataset(HAZARD_ZELL)
xmin, xmax = float(ds["x"].min()), float(ds["x"].max())
ymin, ymax = float(ds["y"].min()), float(ds["y"].max())

# Optional buffer in meters (recommended to avoid cutting edge effects)
buf = 200
xmin_b, xmax_b = xmin - buf, xmax + buf
ymin_b, ymax_b = ymin - buf, ymax + buf

domain_poly = box(xmin_b, ymin_b, xmax_b, ymax_b)

# --- 3) Subset buildings to the domain (FAST bbox filter) ---
# Since geometry is now "geom_polygon_lv95", we set it active temporarily for spatial ops
gdf_build = gdf_build.set_geometry("geom_polygon_lv95")

gdf_build_sub = gdf_build.cx[xmin_b:xmax_b, ymin_b:ymax_b].copy()

print("Buildings before:", len(gdf_build))
print("Buildings after bbox subset:", len(gdf_build_sub))

# --- 4) (Optional) Exact clip to domain bbox polygon (slower, more exact) ---
# If you want to physically cut polygons at the boundary:
# domain_gdf = gpd.GeoDataFrame(geometry=[domain_poly], crs="EPSG:2056")
# gdf_build_sub = gpd.clip(gdf_build_sub, domain_gdf)

# Keep the subset as your main object
gdf_build = gdf_build_sub

# Done: gdf_build now contains only buildings in your model domain
gdf_build.head()


Buildings before: 2154741
Buildings after bbox subset: 1568


,id_def,objectid,geom_polygon_lv95
13166,627672,627672.0,"POLYGON ((2705726.421 1255506.534, 2705725.835..."
13168,627870,627870.0,"POLYGON ((2704016.482 1255528.546, 2704018.014..."
13205,629980,629980.0,"POLYGON ((2706124.285 1255760.375, 2706124.357..."
13270,633463,633463.0,"POLYGON ((2704434.993 1256241.039, 2704435.319..."
13301,635820,635820.0,"POLYGON ((2702715.312 1256554.763, 2702716.035..."


In [46]:
# 1) make inside-points as geometry (LV95)
gdf_build["geometry"] = gdf_build["geom_polygon_lv95"].centroid
gdf_build = gdf_build.set_geometry("geometry")

# 4) Reproject points to WGS84 for CLIMADA; also keep polygons in WGS84 if you want
gdf_build = gdf_build.to_crs(4326)
gdf_build["geom_polygon_wgs84"] = gdf_build["geom_polygon_lv95"].to_crs(4326)

In [47]:
gdf_build = gdf_build.set_geometry("geometry")
gdf_build

,id_def,objectid,geom_polygon_lv95,geometry,geom_polygon_wgs84
13166,627672,627672.0,"POLYGON ((2705726.421 1255506.534, 2705725.835...",POINT (8.84023 47.44187),"POLYGON ((8.84035 47.44186, 8.84034 47.44182, ..."
13168,627870,627870.0,"POLYGON ((2704016.482 1255528.546, 2704018.014...",POINT (8.81766 47.44228),"POLYGON ((8.81769 47.44233, 8.81771 47.44232, ..."
13205,629980,629980.0,"POLYGON ((2706124.285 1255760.375, 2706124.357...",POINT (8.84566 47.44403),"POLYGON ((8.84568 47.44408, 8.84568 47.44405, ..."
13270,633463,633463.0,"POLYGON ((2704434.993 1256241.039, 2704435.319...",POINT (8.82333 47.44868),"POLYGON ((8.82340 47.44867, 8.82340 47.44863, ..."
13301,635820,635820.0,"POLYGON ((2702715.312 1256554.763, 2702716.035...",POINT (8.80036 47.45169),"POLYGON ((8.80067 47.45176, 8.80068 47.45173, ..."
...,...,...,...,...,...
2144513,1279188,1279188.0,"POLYGON ((2702812.763 1256573.584, 2702810.857...",POINT (8.80180 47.45186),"POLYGON ((8.80197 47.45191, 8.80194 47.45189, ..."
2144565,1284180,1284180.0,"POLYGON ((2705057.558 1255776.143, 2705054.464...",POINT (8.83151 47.44439),"POLYGON ((8.83154 47.44439, 8.83150 47.44436, ..."
2144594,1283854,1283854.0,"POLYGON ((2704184.851 1255563.419, 2704183.195...",POINT (8.81978 47.44269),"POLYGON ((8.81993 47.44261, 8.81990 47.44259, ..."
2144595,1283958,1283958.0,"POLYGON ((2704401.541 1255113.069, 2704400.540...",POINT (8.82256 47.43855),"POLYGON ((8.82269 47.43853, 8.82268 47.43849, ..."


In [48]:
import geopandas as gpd
import xarray as xr
from shapely.geometry import box

roads = "/home/gespejogutierrez/Lisflood_climada/Data_for_process/geo_road_arcs.gpkg"
HAZARD_ZELL = "/home/gespejogutierrez/Lisflood_climada/Zell_netcdfiles/Zell_2m_COSMO_2022-05-05T12-00.nc"

# 1) domain bbox from NetCDF (LV95)
ds = xr.open_dataset(HAZARD_ZELL)
x = ds["x"].values
y = ds["y"].values
xmin, xmax = float(x.min()), float(x.max())
ymin, ymax = float(y.min()), float(y.max())

# optional buffer (meters)
buf = 200
domain_poly = box(xmin-buf, ymin-buf, xmax+buf, ymax+buf)

# 2) read roads
gdf_roads = gpd.read_file(roads)

# 3) ensure LV95 CRS
if gdf_roads.crs is None:
    gdf_roads = gdf_roads.set_crs(2056, allow_override=True)
else:
    gdf_roads = gdf_roads.to_crs(2056)

# 4) subset immediately (fast bbox subset)
gdf_roads = gdf_roads.cx[xmin-buf:xmax+buf, ymin-buf:ymax+buf].copy()

print("roads after subset:", len(gdf_roads))
gdf_roads.head()


roads after subset: 694


,id,objectid,uuid,datum_aenderung,datum_erstellung,erstellung_jahr,erstellung_monat,revision_jahr,revision_monat,grund_aenderung,...,y_begin,x_end,y_end,z_begin,z_end,z_min,z_max,nodeid1,nodeid2,geometry
13616,13611,13611,{D61342A7-4E76-4663-94A2-3D6A94BE7E7F},2016-02-02 10:31:08,2007-09-20,2002.0,NaN,2014.0,6.0,400,...,1256141.884,2702132.216,1256097.876,507.240,507.644,507.240,507.644,617674,617622,MULTILINESTRING Z ((2702147.659 1256141.884 50...
13618,13613,13613,{5D93EE21-78CF-4A26-BEFD-029F9B8DAD0E},2016-02-02 09:13:46,2007-09-20,2002.0,NaN,2014.0,6.0,400,...,1256900.286,2701939.566,1256749.481,499.548,501.449,499.548,501.525,616184,616892,MULTILINESTRING Z ((2701744.074 1256900.286 49...
13619,13614,13614,{628A665B-4633-42EC-875F-058382B69630},2016-02-02 10:31:08,2007-09-20,2002.0,NaN,2014.0,6.0,400,...,1256012.455,2702075.518,1255979.981,508.928,509.149,508.928,509.194,617477,617414,MULTILINESTRING Z ((2702092.129 1256012.455 50...
13620,13615,13615,{431AF2B0-EC63-48DF-AE89-E11FEB137D73},2016-02-02 10:31:08,2007-09-20,2002.0,NaN,2014.0,6.0,400,...,1255979.981,2702119.691,1255893.171,509.149,510.234,509.149,510.254,617414,617572,MULTILINESTRING Z ((2702075.518 1255979.981 50...
13621,13616,13616,{C79E22F8-79F1-4D03-A028-D87BC00CE3B1},2016-02-02 10:31:08,2007-09-20,2002.0,NaN,2014.0,6.0,400,...,1256454.010,2702209.289,1256369.576,504.435,504.767,504.435,504.899,618119,617910,MULTILINESTRING Z ((2702270.949 1256454.010 50...


In [49]:
# assume gdf_roads has geometry in LV95 (EPSG:2056)

# 1) rename the active geometry (LV95)
gdf_roads = gdf_roads.rename_geometry("geom_multipolygon_lv95")

# 2) create a WGS84 (EPSG:4326) geometry column
gdf_roads["geometry"] = (
    gdf_roads.set_geometry("geom_multipolygon_lv95")
             .to_crs(4326)
             .geometry
)

#  3) MAKE the WGS84 geometry the ACTIVE geometry
gdf_roads = gdf_roads.set_geometry("geometry")

#  4) FIX the CRS metadata to match the active geometry
gdf_roads = gdf_roads.set_crs(4326, allow_override=True)


In [50]:
def exposure_from_points(gdf, impf_dict):
    # TODO: check why 'FL' tag not taken when read in raster flood file
    exp_pnt = Exposures(gdf)
    exp_pnt.gdf[f'impf_FL'] = getattr(ImpFuncsCIFloodCH(), impf_dict['FL']).id
    exp_pnt.gdf['value'] = 1
    exp_pnt.set_lat_lon()
    exp_pnt.check()
    return exp_pnt
      
def exposure_from_lines(gdf, impf_dict, res, 
                        disagg_met=u_lp.DisaggMethod.FIX, disagg_val=None):
    exp_line = Exposures(gdf)
    if not disagg_val:
        disagg_val = res
    exp_pnt = u_lp.exp_geom_to_pnt(exp_line, res=res, to_meters=True, 
                                   disagg_met=disagg_met, disagg_val=disagg_val)  
    exp_pnt.gdf[f'impf_FL'] = getattr(ImpFuncsCIFloodCH(), impf_dict['FL']).id
    exp_pnt.set_lat_lon()
    exp_pnt.check() 
    return exp_pnt

In [51]:
# some impact functions we have used in the past.
class ImpFuncsCIFloodCH():

    def __init__(self):
        self.tag = 'FL'
        self.building =self.regional_cal()
        self.road = self.regional_cal()

    def regional_cal(self):
        step_impf = ImpactFunc()
        step_impf.id = 1
        step_impf.haz_type = 'FL'
        step_impf.name = 'regional calibration'
        step_impf.intensity_unit = ''
        step_impf.intensity = np.array([0,0.04,1,2,3,4])
        step_impf.mdd = np.array([0,0.03,0.13,0.29,0.48,0.76])
        step_impf.paa = np.sort(np.linspace(1, 1, num=6))
        step_impf.check()
        return step_impf

In [52]:
import copy
exp_building = exposure_from_points(gdf_build, {'FL': 'building'})
exp_rd = exposure_from_lines(copy.deepcopy(gdf_roads), {'FL': 'road'}, res=500)

In [53]:
def make_impfset(imp_class):
    impfset = ImpactFuncSet()
    for attribute in set(imp_class.__dict__.keys()).difference({'tag'}):
        impfset.append(getattr(imp_class, attribute))
    return impfset

In [54]:
impf_set_fl = make_impfset(ImpFuncsCIFloodCH())

In [55]:
haz_names ['ZELL_2022-05-05T19:00'].intensity[7].data

array([1.363, 1.274, 1.182, ..., 0.138, 0.034, 0.051], dtype=float32)

In [ ]:
## IMPACT 

In [56]:
haz = haz_names['ZELL_2022-05-05T19:00']
print("Intensity shape:", haz.intensity.shape)
print("Number of event names:", len(haz.event_name))
print("Event names:", haz.event_name)
print("Event IDs:", haz.event_id)

Intensity shape: (11, 3750000)
Number of event names: 11
Event names: ['ZELL_ens01_2022-05-05T19:00' 'ZELL_ens02_2022-05-05T19:00'
 'ZELL_ens03_2022-05-05T19:00' 'ZELL_ens04_2022-05-05T19:00'
 'ZELL_ens05_2022-05-05T19:00' 'ZELL_ens06_2022-05-05T19:00'
 'ZELL_ens07_2022-05-05T19:00' 'ZELL_ens08_2022-05-05T19:00'
 'ZELL_ens09_2022-05-05T19:00' 'ZELL_ens10_2022-05-05T19:00'
 'ZELL_ens11_2022-05-05T19:00']
Event IDs: [ 1  2  3  4  5  6  7  8  9 10 11]


In [ ]:
from climada.engine import Impact

impbuilding = {}

for haz_key, haz_obj in haz_names.items():
    print(f"Calculating impacts for {haz_key} ...")
    imp = Impact()
    imp.calc(exp_building, impf_set_fl, haz_obj, save_mat=True)
    impbuilding[haz_key] = imp

print(f"\n Created {len(impbuilding)} Impact objects (directly from CLIMADA-compliant hazards).")

In [38]:
impbuilding ['ZELL_2022-05-05T15:00'].imp_mat

<11x1568 sparse matrix of type '<class 'numpy.float64'>'
	with 9 stored elements in Compressed Sparse Row format>

In [ ]:
import pickle
import os

out_path = "/home/gespejogutierrez/Lisflood_climada/picklefiles/impbuilding_Zell_forecast.pkl"
os.makedirs(os.path.dirname(out_path), exist_ok=True)

# Save the FULL dictionary exactly as-is
with open(out_path, "wb") as f:
    pickle.dump(
        impbuilding,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("✔ Saved:", out_path)
print("Saved object type:", type(impbuilding))
print("Number of keys:", len(impbuilding))
print("Keys:", list(impbuilding.keys()))

✔ Saved: /home/gespejogutierrez/Lisflood_climada/picklefiles/impbuilding_Zell_forecast.pkl
Saved object type: <class 'dict'>
Number of keys: 11
Keys: ['ZELL_2022-05-05T12:00', 'ZELL_2022-05-05T13:00', 'ZELL_2022-05-05T14:00', 'ZELL_2022-05-05T15:00', 'ZELL_2022-05-05T16:00', 'ZELL_2022-05-05T17:00', 'ZELL_2022-05-05T18:00', 'ZELL_2022-05-05T19:00', 'ZELL_2022-05-05T20:00', 'ZELL_2022-05-05T21:00', 'ZELL_2022-05-05T22:00']


In [60]:
from climada.engine import Impact

improad = {}

for haz_key, haz_obj in haz_names.items():
    print(f"Calculating impacts for {haz_key} ...")
    imprd = Impact()
    imprd.calc(exp_rd, impf_set_fl, haz_obj, save_mat=True)
    improad[haz_key] = imprd

print(f"\n Created {len(improad)} Impact objects (directly from CLIMADA-compliant hazards).")

Calculating impacts for ZELL_2022-05-05T12:00 ...
2025-12-21 20:31:30,865 - climada.engine.impact - WARNING - The use of Impact().calc() is deprecated. Use ImpactCalc().impact() instead.
Calculating impacts for ZELL_2022-05-05T13:00 ...
2025-12-21 20:31:34,576 - climada.engine.impact - WARNING - The use of Impact().calc() is deprecated. Use ImpactCalc().impact() instead.
Calculating impacts for ZELL_2022-05-05T14:00 ...
2025-12-21 20:31:38,258 - climada.engine.impact - WARNING - The use of Impact().calc() is deprecated. Use ImpactCalc().impact() instead.
Calculating impacts for ZELL_2022-05-05T15:00 ...
2025-12-21 20:31:42,066 - climada.engine.impact - WARNING - The use of Impact().calc() is deprecated. Use ImpactCalc().impact() instead.
Calculating impacts for ZELL_2022-05-05T16:00 ...
2025-12-21 20:31:45,761 - climada.engine.impact - WARNING - The use of Impact().calc() is deprecated. Use ImpactCalc().impact() instead.
Calculating impacts for ZELL_2022-05-05T17:00 ...
2025-12-21 20:3

In [63]:
improad ['ZELL_2022-05-05T12:00'].imp_mat

<11x698 sparse matrix of type '<class 'numpy.float64'>'
	with 0 stored elements in Compressed Sparse Row format>

In [64]:
import pickle
import os

out_path = "/home/gespejogutierrez/Lisflood_climada/picklefiles/improad_Zell_forecast.pkl"
os.makedirs(os.path.dirname(out_path), exist_ok=True)

# Save the FULL dictionary exactly as-is
with open(out_path, "wb") as f:
    pickle.dump(
        improad,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )

print("✔ Saved:", out_path)
print("Saved object type:", type(improad))
print("Number of keys:", len(improad))
print("Keys:", list(improad.keys()))

✔ Saved: /home/gespejogutierrez/Lisflood_climada/picklefiles/improad_Zell_forecast.pkl
Saved object type: <class 'dict'>
Number of keys: 11
Keys: ['ZELL_2022-05-05T12:00', 'ZELL_2022-05-05T13:00', 'ZELL_2022-05-05T14:00', 'ZELL_2022-05-05T15:00', 'ZELL_2022-05-05T16:00', 'ZELL_2022-05-05T17:00', 'ZELL_2022-05-05T18:00', 'ZELL_2022-05-05T19:00', 'ZELL_2022-05-05T20:00', 'ZELL_2022-05-05T21:00', 'ZELL_2022-05-05T22:00']


In [ ]:
#### More Analysis 

In [ ]:
import numpy as np
import geopandas as gpd


def impact_to_gdf_wide(
    imp,
    col_prefix="ens",          # ens0, ens1, ...
    add_exp_id=True,
    gdf_build=None,            # must contain 'geom_polygon_lv95'
):
    """
    Wide GeoDataFrame for ONE Impact object (one forecast time).

    Output:
      - geometry  : building polygons from gdf_build['geom_polygon_lv95'] (EPSG:2056)
      - lat, lon  : CLIMADA exposure coordinates (EPSG:4326)
      - ens0, ens1, ... : impact for each event / ensemble member
    """

    if gdf_build is None:
        raise ValueError("gdf_build (with 'geom_polygon_lv95') must be provided.")

    if "geom_polygon_lv95" not in gdf_build.columns:
        raise ValueError("gdf_build must have a 'geom_polygon_lv95' column.")

    n_events, n_exp = imp.imp_mat.shape

    if len(gdf_build) != n_exp:
        raise ValueError(
            f"gdf_build has {len(gdf_build)} rows but impact has {n_exp} exposures. "
            "They must be in the same order."
        )

    # ---------- start from building polygons ----------
    gdf = gdf_build.copy()

    # make sure polygon column is the active geometry in LV95
    gdf = gpd.GeoDataFrame(
        gdf,
        geometry="geom_polygon_lv95",
        crs="EPSG:2056",
    )

    # ---------- add CLIMADA exposure coordinates (lat, lon) ----------
    lats = imp.coord_exp[:, 0]
    lons = imp.coord_exp[:, 1]

    if len(lats) != n_exp:
        raise ValueError("Number of coord_exp points does not match number of exposures.")

    gdf["lat"] = lats
    gdf["lon"] = lons

    # optional exp_id
    if add_exp_id and "exp_id" not in gdf.columns:
        gdf["exp_id"] = np.arange(n_exp, dtype=int)

    # ---------- impacts: one column per ensemble member ----------
    mat = imp.imp_mat.tocsr()
    for i in range(n_events):
        gdf[f"{col_prefix}{i}"] = mat.getrow(i).toarray().ravel()

    return gdf


In [ ]:
# impbuilding: dict like {'ZELL_2022-05-05T19:00': Impact, ...}

gdf_wide = {}   # this will be your dict of GeoDataFrames

for time_key, imp in impbuilding.items():
    gdf_wide[time_key] = impact_to_gdf_wide(
        imp,
        col_prefix="ens",
        add_exp_id=True,
        gdf_build=gdf_build,   # your table with geom_polygon_lv95
    )

In [ ]:
gdf_wide ['ZELL_2022-05-05T12:00']

,id_def,objectid,geom_polygon_lv95,geometry,geom_polygon_wgs84,lat,lon,exp_id,ens0,ens1,ens2,ens3,ens4,ens5,ens6,ens7,ens8,ens9,ens10
13166,627672,627672.0,"POLYGON ((2705726.421 1255506.534, 2705725.835...",POINT (8.84023 47.44187),"POLYGON ((8.84035 47.44186, 8.84034 47.44182, ...",47.441870,8.840233,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13168,627870,627870.0,"POLYGON ((2704016.482 1255528.546, 2704018.014...",POINT (8.81766 47.44228),"POLYGON ((8.81769 47.44233, 8.81771 47.44232, ...",47.442282,8.817663,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13205,629980,629980.0,"POLYGON ((2706124.285 1255760.375, 2706124.357...",POINT (8.84566 47.44403),"POLYGON ((8.84568 47.44408, 8.84568 47.44405, ...",47.444032,8.845660,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13270,633463,633463.0,"POLYGON ((2704434.993 1256241.039, 2704435.319...",POINT (8.82333 47.44868),"POLYGON ((8.8234 47.44867, 8.8234 47.44863, 8....",47.448684,8.823325,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13301,635820,635820.0,"POLYGON ((2702715.312 1256554.763, 2702716.035...",POINT (8.80036 47.45169),"POLYGON ((8.80067 47.45176, 8.80068 47.45173, ...",47.451694,8.800357,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2144513,1279188,1279188.0,"POLYGON ((2702812.763 1256573.584, 2702810.857...",POINT (8.8018 47.45186),"POLYGON ((8.80197 47.45191, 8.80194 47.45189, ...",47.451855,8.801795,1563,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2144565,1284180,1284180.0,"POLYGON ((2705057.558 1255776.143, 2705054.464...",POINT (8.83151 47.44439),"POLYGON ((8.83154 47.44439, 8.8315 47.44436, 8...",47.444390,8.831512,1564,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2144594,1283854,1283854.0,"POLYGON ((2704184.851 1255563.419, 2704183.195...",POINT (8.81978 47.44269),"POLYGON ((8.81993 47.44261, 8.8199 47.44259, 8...",47.442693,8.819780,1565,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2144595,1283958,1283958.0,"POLYGON ((2704401.541 1255113.069, 2704400.54 ...",POINT (8.82256 47.43855),"POLYGON ((8.82269 47.43853, 8.82268 47.43849, ...",47.438551,8.822562,1566,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd


def impact_to_gdf_long(
    imp,
    time_key=None,              # e.g. "ZELL_2022-05-05T12:00" (optional)
    gdf_build=None,             # must contain 'geom_polygon_lv95' (and ideally 'exp_id')
    exp_id_col="exp_id",
    geom_col="geom_polygon_lv95",
    event_name_prefix="ens",    # if imp.event_name missing, fallback to ens0, ens1, ...
    keep_point_geometry=False,  # add exposure point geometry (WGS84) as extra column
):
    """
    LONG GeoDataFrame for ONE Impact object (one forecast time).

    Output columns (typical):
      - geometry      : building polygons from gdf_build[geom_col] (EPSG:2056)
      - lat, lon      : CLIMADA exposure coordinates (EPSG:4326)
      - exp_id        : exposure id (from gdf_build if present, else generated)
      - event_id      : 1..n_events
      - event_name    : CLIMADA event name (if available) or ens0, ens1, ...
      - impact        : impact value for (event, exposure)
      - time_key      : your passed key (optional)
      - run_datetime  : parsed datetime from time_key if possible (optional)
    """

    if gdf_build is None:
        raise ValueError("gdf_build (with polygons) must be provided.")
    if geom_col not in gdf_build.columns:
        raise ValueError(f"gdf_build must have a '{geom_col}' column.")

    n_events, n_exp = imp.imp_mat.shape

    # Ensure same number of exposures as polygons table
    if len(gdf_build) != n_exp:
        raise ValueError(
            f"gdf_build has {len(gdf_build)} rows but impact has {n_exp} exposures. "
            "They must match and be in the same order."
        )

    # Base table from polygons
    base = gdf_build.copy()

    # Ensure exp_id exists (stable join key)
    if exp_id_col not in base.columns:
        base[exp_id_col] = np.arange(n_exp, dtype=int)

    # Set polygon geometry (LV95)
    base = gpd.GeoDataFrame(base, geometry=geom_col, crs="EPSG:2056")

    # Add exposure coords (WGS84)
    lats = imp.coord_exp[:, 0]
    lons = imp.coord_exp[:, 1]
    if len(lats) != n_exp:
        raise ValueError("Number of coord_exp points does not match number of exposures.")

    base["lat"] = lats
    base["lon"] = lons

    # Optional: keep exposure point geometry as extra column
    if keep_point_geometry:
        base["geometry_wgs84_pt"] = gpd.points_from_xy(base["lon"], base["lat"], crs="EPSG:4326")

    # Event labels
    if hasattr(imp, "event_name") and imp.event_name is not None and len(imp.event_name) == n_events:
        event_names = list(imp.event_name)
    else:
        event_names = [f"{event_name_prefix}{i}" for i in range(n_events)]

    # Convert sparse -> COO for efficient long format
    mat = imp.imp_mat.tocoo()

    # Create long rows ONLY where impact is non-zero
    df_long = pd.DataFrame({
        "event_idx": mat.row.astype(int),       # 0-based
        "exp_idx":   mat.col.astype(int),       # 0-based
        "impact":    mat.data.astype(float),
    })

    # Map event fields
    df_long["event_id"] = df_long["event_idx"] + 1
    df_long["event_name"] = [event_names[i] for i in df_long["event_idx"].values]

    # Attach exposure ids and coordinates by exp_idx
    df_long[exp_id_col] = base[exp_id_col].values[df_long["exp_idx"].values]
    df_long["lat"] = base["lat"].values[df_long["exp_idx"].values]
    df_long["lon"] = base["lon"].values[df_long["exp_idx"].values]

    # Optional time info
    if time_key is not None:
        df_long["time_key"] = str(time_key)
        # try to parse datetime from key like "ZELL_2022-05-05T12:00"
        try:
            dt_str = str(time_key).split("_", 1)[-1]
            df_long["run_datetime"] = pd.to_datetime(dt_str)
        except Exception:
            pass

    # Now build GeoDataFrame by joining polygons from base using exp_idx
    # We avoid a merge by using direct indexing (fast and preserves order)
    geom = base.geometry.values[df_long["exp_idx"].values]
    gdf_long = gpd.GeoDataFrame(df_long, geometry=geom, crs="EPSG:2056")

    # Clean helper cols if you don't want them
    gdf_long = gdf_long.drop(columns=["event_idx", "exp_idx"])

    return gdf_long


In [ ]:
# Example usage with your dict of impacts
# -----------------------
gdf_long = {}
for time_key, imp in impbuilding.items():
     gdf_long[time_key] = impact_to_gdf_long(
         imp,
         time_key=time_key,
        gdf_build=gdf_build,
         exp_id_col="exp_id",
         geom_col="geom_polygon_lv95",
         keep_point_geometry=False
     )

In [ ]:
gdf_long ['ZELL_2022-05-05T17:00']

,impact,event_id,event_name,exp_id,lat,lon,time_key,run_datetime,geometry
0,0.046250,3,ZELL_ens03_2022-05-05T17:00,48,47.435405,8.855446,ZELL_2022-05-05T17:00,2022-05-05 17:00:00,"POLYGON ((2706872.658 1254811.771, 2706872.916..."
1,0.030208,3,ZELL_ens03_2022-05-05T17:00,62,47.464562,8.845225,ZELL_2022-05-05T17:00,2022-05-05 17:00:00,"POLYGON ((2706058.413 1258039.474, 2706055.531..."
2,0.012750,3,ZELL_ens03_2022-05-05T17:00,85,47.451415,8.802858,ZELL_2022-05-05T17:00,2022-05-05 17:00:00,"POLYGON ((2702889.276 1256523.343, 2702889.739..."
3,0.010500,3,ZELL_ens03_2022-05-05T17:00,110,47.462847,8.846664,ZELL_2022-05-05T17:00,2022-05-05 17:00:00,"POLYGON ((2706165.851 1257849.069, 2706165.806..."
4,0.010500,3,ZELL_ens03_2022-05-05T17:00,125,47.438056,8.846512,ZELL_2022-05-05T17:00,2022-05-05 17:00:00,"POLYGON ((2706210.608 1255092.027, 2706206.927..."
5,0.030313,3,ZELL_ens03_2022-05-05T17:00,331,47.435957,8.849880,ZELL_2022-05-05T17:00,2022-05-05 17:00:00,"POLYGON ((2706464.41 1254872.769, 2706463.182 ..."
6,0.016500,3,ZELL_ens03_2022-05-05T17:00,341,47.436270,8.855717,ZELL_2022-05-05T17:00,2022-05-05 17:00:00,"POLYGON ((2706889.05 1254901, 2706886.35 12549..."
7,0.031042,3,ZELL_ens03_2022-05-05T17:00,414,47.435037,8.856905,ZELL_2022-05-05T17:00,2022-05-05 17:00:00,"POLYGON ((2706993.785 1254777.219, 2706994.061..."
8,0.030417,3,ZELL_ens03_2022-05-05T17:00,804,47.436959,8.851021,ZELL_2022-05-05T17:00,2022-05-05 17:00:00,"POLYGON ((2706546.483 1254979.245, 2706545.512..."
9,0.030313,3,ZELL_ens03_2022-05-05T17:00,857,47.437110,8.852845,ZELL_2022-05-05T17:00,2022-05-05 17:00:00,"POLYGON ((2706684.366 1255000.178, 2706683.66 ..."
